[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CrashingGuru/ITUAIReadiness/blob/main/Colab%20Free%20Tier%20LLM%20Notebook.ipynb)

# AI for Good: Local LLM + Knowledge Base Notebook
Authors: Vishnu & Jiaying.  
Date: 22Mar2026.  

This notebook is designed to run **directly in Google Colab** using the **free tier**.

It builds a simple local-RAG workflow:
1. Install all dependencies first
2. Verify the runtime
3. Configure paths and settings
4. Mount Google Drive for persistence
5. Download a small local GGUF model
6. Load documents (`.pdf`, `.txt`, `.md`)
7. Chunk and embed them
8. Build or reload a FAISS index
9. Run retrieval + local generation
10. Ask questions with one cell

> **Note on cell numbers:** The step numbers in comments (`# Cell 2`, `# Cell 3`, …) and in the *Recommended run order* section are **logical step numbers**, not Jupyter's UI cell indices. Jupyter numbers cells sequentially as you add them; these labels refer to the order you should execute the workflow steps.

## Reference links

- Google Colab FAQ: https://research.google.com/colaboratory/faq.html
- Colab Drive I/O example: https://colab.research.google.com/notebooks/io.ipynb
- llama.cpp: https://github.com/ggml-org/llama.cpp
- llama-cpp-python: https://github.com/abetlen/llama-cpp-python
- llama-cpp-python docs: https://llama-cpp-python.readthedocs.io/en/latest/
- GGUF + llama.cpp on Hugging Face: https://huggingface.co/docs/hub/gguf-llamacpp
- Llama-3.2-1B-Instruct GGUF (bartowski): https://huggingface.co/bartowski/Llama-3.2-1B-Instruct-GGUF
- TinyLlama GGUF (fallback): https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF
- Sentence Transformers pretrained models: https://www.sbert.net/docs/sentence_transformer/pretrained_models.html
- all-MiniLM-L6-v2 model card: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
- Hugging Face Hub download docs: https://huggingface.co/docs/huggingface_hub/en/guides/download
- FAISS: https://github.com/facebookresearch/faiss
- pypdf: https://pypdf.readthedocs.io/

## Credential notes

- **Google Drive authorization is required later when you mount Drive.**
- **You will use Google's standard authorization flow. Do not paste your Google password into notebook code.**
- **The default public model download does not require a Hugging Face token.**
- **If you later switch to a gated/private model, you may need a Hugging Face access token.**

**Important:** The first time you run the install cell, it may automatically restart the Colab runtime after locking package versions. This is expected. After the restart, continue from the import/verification cell.


In [ ]:
# ============================================================
# Cell 2 - Install missing dependencies
#
# Run this cell once at the start of each new Colab session.
#
# After installation completes you will see a box asking you to
# restart the runtime. Click the button or use:
#   Runtime → Restart session
# Then re-run this cell (it will say 'already installed') and
# continue with Cell 3 onwards.
# ============================================================

import sys
import subprocess

def run(cmd):
    print(">>", " ".join(cmd))
    subprocess.check_call(cmd)

def _all_installed():
    """True only when every required package is importable."""
    try:
        import faiss          # noqa: F401
        import pypdf           # noqa: F401
        from llama_cpp import Llama  # noqa: F401
        return True
    except (ImportError, ModuleNotFoundError):
        return False

if _all_installed():
    print("✅ All required packages already importable. Continue to Cell 3.")

else:
    print("Installing missing packages...\n")

    # ── Detect torch build tag (e.g. 'cu128') ────────────────────────────────
    _r = subprocess.run(
        [sys.executable, "-c",
         "import torch; v=torch.__version__; print(v.split('+')[1] if '+' in v else 'cpu')"],
        capture_output=True, text=True
    )
    _build_tag = _r.stdout.strip()
    print(f"Torch build tag: {_build_tag}\n")

    # ── Realign torchvision to match torch build tag ──────────────────────────
    # --no-deps: only reinstall torchvision itself, not the full torch stack.
    if _build_tag.startswith("cu"):
        print("Reinstalling torchvision (--no-deps to skip torch re-download)...")
        run([sys.executable, "-m", "pip", "install", "-q",
             "--force-reinstall", "--no-deps",
             "torchvision",
             "--index-url", f"https://download.pytorch.org/whl/{_build_tag}"])
    else:
        print("CPU runtime — skipping torchvision realignment.")

    # ── faiss-cpu and pypdf ───────────────────────────────────────────────────
    print("\nInstalling faiss-cpu and pypdf...")
    run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "pypdf>=4.0"])

    # ── llama-cpp-python from pre-built wheel ─────────────────────────────────
    # --only-binary :all: → fail fast if no wheel found; never compile from
    # source (source compile takes 8-10 min and OOM-kills Colab free tier).
    print("\nInstalling llama-cpp-python from pre-built wheel...")
    _llama_ok = False
    for _tag in ([_build_tag, "cpu"] if _build_tag.startswith("cu") else ["cpu"]):
        _index = f"https://abetlen.github.io/llama-cpp-python/whl/{_tag}"
        print(f"  Trying '{_tag}' wheel...")
        _r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "--only-binary", ":all:",
             "llama-cpp-python",
             "--extra-index-url", _index],
            capture_output=True, text=True
        )
        if _r.returncode == 0:
            print(f"  ✅ Installed ({_tag} wheel).")
            _llama_ok = True
            break
        print(f"  No binary wheel for '{_tag}'.")

    if not _llama_ok:
        print("\n⚠️  No pre-built wheel found. Falling back to source compile.")
        print("    This will take ~8-10 min. If it crashes, switch to a CPU runtime.")
        run([sys.executable, "-m", "pip", "install", "llama-cpp-python"])

    # ── Prompt for manual restart ─────────────────────────────────────────────
    # We do NOT call os.kill here. SIGKILL looks identical to an unexpected
    # crash and confuses users. Instead we prompt for a manual restart and
    # use the import check above to skip re-installation after the restart.
    print("\n" + "═" * 58)
    print("  ✅  Installation complete!")
    print("")
    print("  ➡️   You must now restart the runtime:")
    print("        Runtime  →  Restart session")
    print("")
    print("  After restarting, re-run this cell (it will skip the")
    print("  install) then continue from Cell 3 onwards.")
    print("═" * 58)

    # Trigger Colab's built-in restart dialog (shows a UI button, not a crash)
    try:
        from google.colab.output import eval_js
        eval_js('google.colab.kernel.restart()')
    except Exception:
        pass  # Outside Colab or already restarting — the printed message is enough


✅ Locked dependencies already installed for this runtime. Continuing without reinstall.


In [ ]:
# ============================================================
# Cell 3 - Import libraries and verify environment
# ============================================================

import os
import sys
import json
import math
import re
import time
import pickle
import shutil
import platform
from pathlib import Path

import numpy as np
import faiss
import pypdf
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from llama_cpp import Llama

# Catch both ImportError and RuntimeError (the latter occurs when torch and
# torchvision were compiled against different CUDA versions).
try:
    import torch
    # Force an actual CUDA check to surface any version-mismatch RuntimeError now.
    _ = torch.cuda.is_available()
    TORCH_AVAILABLE = True
except (ImportError, RuntimeError) as _torch_err:
    print(f"⚠️  torch unavailable: {_torch_err}")
    print("    GPU offload will be disabled. Re-run Cell 2 to fix the CUDA version mismatch.")
    TORCH_AVAILABLE = False

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())
print("NumPy version:", np.__version__)
print("Torch available:", TORCH_AVAILABLE)
if TORCH_AVAILABLE:
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device:", torch.cuda.get_device_name(0))
        print("torch CUDA version:", torch.version.cuda)
    else:
        print("CUDA device: none (CPU runtime or no visible GPU)")
print("llama_cpp imported from:", Llama.__module__)
print("✅ Imports verified")


/usr/local/lib/python3.12/dist-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from this module in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Working directory: /content
NumPy version: 1.26.4
NumPy char module OK: True
Torch available: True
CUDA available: True
CUDA device: Tesla T4
✅ Imports verified


In [ ]:
import json

# ============================================================
# Cell 4 - Configuration
# Edit values here before running the main workflow
# ============================================================

CONFIG = {
    # Drive project folder
    "project_root": "/content/drive/MyDrive/colab_local_llm_kb",

    # Public GGUF model — bartowski/Llama-3.2-1B-Instruct-GGUF is a strong
    # free-tier choice (~700 MB Q4_K_M). Swap model_repo_id + model_filename
    # to use a different model; no other changes required.
    "model_repo_id": "bartowski/Llama-3.2-1B-Instruct-GGUF",
    "model_filename": "Llama-3.2-1B-Instruct-Q4_K_M.gguf",

    # Embedding model
    "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",

    # Data folders
    "docs_dirname": "docs",
    "models_dirname": "models",
    "index_dirname": "index",

    # Parsing / chunking
    "supported_extensions": [".pdf", ".txt", ".md"],
    "chunk_size": 800,
    "chunk_overlap": 120,
    "min_chunk_chars": 80,

    # Retrieval / generation
    "top_k": 4,
    "max_tokens": 256,
    "temperature": 0.2,
    "n_ctx": 4096,   # increased from 2048; RAG with 4 x 800-char chunks needs headroom
    "n_threads": 4,  # CPU threads for llama-cpp inference; 4 is a safe default on Colab

    # Workflow mode
    # True  = parse docs and rebuild the FAISS index from scratch
    # False = skip parsing and reload a previously saved index from Drive
    "rebuild_index": True,
    "allow_local_uploads": True,
}

print(json.dumps(CONFIG, indent=2))
print("✅ Configuration loaded")


{
  "project_root": "/content/drive/MyDrive/colab_local_llm_kb",
  "model_repo_id": "TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
  "model_filename": "tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "docs_dirname": "docs",
  "models_dirname": "models",
  "index_dirname": "index",
  "supported_extensions": [
    ".pdf",
    ".txt",
    ".md"
  ],
  "chunk_size": 800,
  "chunk_overlap": 120,
  "min_chunk_chars": 80,
  "top_k": 4,
  "max_tokens": 256,
  "temperature": 0.2,
  "n_ctx": 2048,
  "rebuild_index": true,
  "allow_local_uploads": true
}
✅ Configuration loaded


## Main execution starts here

### Google Drive authorization
**Highlighted note:** this step will trigger Google's standard Drive authorization flow.

- You will sign in through Google's normal popup/consent process.
- **Do not type your Google username/password into notebook code cells.**
- Drive is used so your model file and FAISS index survive runtime resets.

In [ ]:
# ============================================================
# Cell 5 - Mount Google Drive and create folders
# Reference:
# https://colab.research.google.com/notebooks/io.ipynb
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path(CONFIG["project_root"])
MODELS_DIR = PROJECT_ROOT / CONFIG["models_dirname"]
DOCS_DIR = PROJECT_ROOT / CONFIG["docs_dirname"]
INDEX_DIR = PROJECT_ROOT / CONFIG["index_dirname"]

for folder in [PROJECT_ROOT, MODELS_DIR, DOCS_DIR, INDEX_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("MODELS_DIR   =", MODELS_DIR)
print("DOCS_DIR     =", DOCS_DIR)
print("INDEX_DIR    =", INDEX_DIR)
print("✅ Drive mounted and folders prepared")

Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/colab_local_llm_kb
MODELS_DIR   = /content/drive/MyDrive/colab_local_llm_kb/models
DOCS_DIR     = /content/drive/MyDrive/colab_local_llm_kb/docs
INDEX_DIR    = /content/drive/MyDrive/colab_local_llm_kb/index
✅ Drive mounted and folders prepared


### Model download
**Highlighted note:** the default notebook uses a **public** GGUF model and does **not** require a username/password or Hugging Face token.

If you later replace it with a gated/private model, you may need a Hugging Face token.

In [ ]:
# ============================================================
# Cell 6 - Download the local GGUF model
# Why:
# - Llama-3.2-1B-Instruct is small enough for Colab free tier (~700 MB Q4_K_M)
# - GGUF is the standard format for llama.cpp/llama-cpp-python
# - bartowski is the maintained GGUF provider for modern models
# References:
# https://huggingface.co/bartowski/Llama-3.2-1B-Instruct-GGUF
# https://huggingface.co/docs/hub/gguf-llamacpp
# ============================================================

MODEL_PATH = MODELS_DIR / CONFIG["model_filename"]

if MODEL_PATH.exists():
    print(f"✅ Model already present: {MODEL_PATH}")
else:
    print("Downloading GGUF model from Hugging Face Hub...")
    downloaded_path = hf_hub_download(
        repo_id=CONFIG["model_repo_id"],
        filename=CONFIG["model_filename"],
        local_dir=str(MODELS_DIR),
    )
    print("Downloaded to:", downloaded_path)

print("Final model path:", MODEL_PATH)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf:   0%|          | 0.00/669M [00:00<?, ?B/s]

Downloaded to: /content/drive/MyDrive/colab_local_llm_kb/models/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf
Final model path: /content/drive/MyDrive/colab_local_llm_kb/models/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf


## Documents input

This notebook supports:
- `.pdf`
- `.txt`
- `.md`

You can either upload files from your computer or place them directly in the Drive `docs` folder.

**No username/password is needed in this section.**

In [ ]:
# ============================================================
# Cell 7 - Upload local documents (optional)
# If you prefer, skip this cell and manually copy files into DOCS_DIR in Drive.
# ============================================================

if CONFIG["allow_local_uploads"]:
    from google.colab import files

    print("Upload .pdf, .txt, or .md files. Cancel if you want to use only Drive files.")
    uploaded = files.upload()

    for name, data in uploaded.items():
        target_path = DOCS_DIR / name
        with open(target_path, "wb") as f:
            f.write(data)
        print("Saved:", target_path)

    print("✅ Upload step finished")
else:
    print("Local uploads disabled by config")

Upload .pdf, .txt, or .md files. Cancel if you want to use only Drive files.


Saving AI Ready Framework 2025.pdf to AI Ready Framework 2025.pdf
Saved: /content/drive/MyDrive/colab_local_llm_kb/docs/AI Ready Framework 2025.pdf
✅ Upload step finished


In [ ]:
# ============================================================
# Cell 8 - Parse, clean, and chunk documents
# Why chunking:
# - Embedding models like all-MiniLM-L6-v2 are better suited to shorter spans
# - Chunking improves retrieval granularity
# ============================================================

from typing import List

def read_pdf_text(path: Path) -> str:
    text_parts = []
    reader = pypdf.PdfReader(str(path))
    for page_num, page in enumerate(reader.pages):
        try:
            text_parts.append(page.extract_text() or "")
        except Exception as e:
            text_parts.append(f"\n[Page {page_num+1} extraction error: {e}]\n")
    return "\n".join(text_parts)

def read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")

def normalize_text(text: str) -> str:
    """Normalize whitespace while preserving paragraph boundaries."""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)   # collapse 3+ newlines to paragraph break
    text = re.sub(r"[^\S\n]+", " ", text)     # collapse spaces/tabs, preserve newlines
    return text.strip()

def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    """Split text into overlapping chunks, breaking at word boundaries."""
    text = text.strip()
    if not text:
        return []
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + chunk_size, n)
        if end < n:
            # walk back to last whitespace within the final 10% of the chunk
            ws = text.rfind(" ", start + int(chunk_size * 0.9), end)
            if ws != -1:
                end = ws
        chunk = text[start:end].strip()
        if len(chunk) >= CONFIG["min_chunk_chars"]:
            chunks.append(chunk)
        if end >= n:
            break
        start = max(end - overlap, start + 1)
    return chunks

documents = []
supported = set(CONFIG["supported_extensions"])

for path in sorted(DOCS_DIR.glob("*")):
    if path.suffix.lower() not in supported:
        continue

    if path.suffix.lower() == ".pdf":
        raw_text = read_pdf_text(path)
    else:
        raw_text = read_text_file(path)

    clean_text = normalize_text(raw_text)
    chunks = chunk_text(clean_text, CONFIG["chunk_size"], CONFIG["chunk_overlap"])

    for idx, chunk in enumerate(chunks):
        documents.append({
            "source": path.name,
            "chunk_id": idx,
            "text": chunk,
        })

print(f"Loaded {len(documents)} chunks from {DOCS_DIR}")
if documents:
    print("Sample source:", documents[0]["source"])
    print("Sample chunk preview:", documents[0]["text"][:300], "...")
else:
    print("No documents found. Add files to DOCS_DIR or rerun the upload cell.")


Loaded 182 chunks from /content/drive/MyDrive/colab_local_llm_kb/docs
Sample source: AI Ready Framework 2025.pdf
Sample chunk preview: ITUPublicationsInternational Telecommunication Union
Telecommunication Standardization Sector
AI Ready – Analysis Towards
a Standardized Readiness
Framework
Report 2.0
January 2026
Disclaimer
The views expressed in this publication are those of the authors and do not necessarily reflect
the views of ...


In [ ]:
# ============================================================
# Cell 9 - Build or reload the FAISS index
# Controlled by CONFIG["rebuild_index"]:
#   True  = embed documents and build a new index
#   False = reload a previously saved index from Drive
# Why:
# - sentence-transformers/all-MiniLM-L6-v2 is lightweight and widely used
# - FAISS provides local vector search with no external service
# References:
# https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
# https://github.com/facebookresearch/faiss
# ============================================================

INDEX_FILE  = INDEX_DIR / "faiss.index"
META_FILE   = INDEX_DIR / "metadata.pkl"
CONFIG_FILE = INDEX_DIR / "config.json"

DEVICE = "cuda" if TORCH_AVAILABLE and torch.cuda.is_available() else "cpu"

if CONFIG["rebuild_index"]:
    if not documents:
        raise ValueError("No document chunks available. Add documents and rerun Cell 8.")

    print("Loading embedding model:", CONFIG["embedding_model_name"], f"(device={DEVICE})")
    embedder = SentenceTransformer(CONFIG["embedding_model_name"], device=DEVICE)

    texts = [d["text"] for d in documents]
    print(f"Encoding {len(texts)} chunks...")
    embeddings = embedder.encode(
        texts,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings.astype("float32"))

    faiss.write_index(index, str(INDEX_FILE))
    with open(META_FILE, "wb") as f:
        pickle.dump(documents, f)
    with open(CONFIG_FILE, "w", encoding="utf-8") as f:
        json.dump(CONFIG, f, indent=2)

    print("✅ Index built and saved")
    print("Index file:", INDEX_FILE)
    print("Metadata file:", META_FILE)

else:
    if not INDEX_FILE.exists() or not META_FILE.exists():
        raise FileNotFoundError("Saved index files not found. Set rebuild_index=True and rerun.")

    index = faiss.read_index(str(INDEX_FILE))
    with open(META_FILE, "rb") as f:
        documents = pickle.load(f)

    # Warn if key config values differ from what was used to build the index
    if CONFIG_FILE.exists():
        with open(CONFIG_FILE) as f:
            saved_cfg = json.load(f)
        for key in ("chunk_size", "embedding_model_name", "chunk_overlap"):
            if saved_cfg.get(key) != CONFIG[key]:
                print(f"⚠️  Config mismatch for '{key}': saved={saved_cfg[key]!r}, current={CONFIG[key]!r}")
                print("    Consider setting rebuild_index=True to rebuild with updated settings.")

    print("✅ Reloaded index and metadata")
    print("Vectors in index:", index.ntotal)
    print("Chunks loaded:", len(documents))

    print("Loading embedding model:", CONFIG["embedding_model_name"], f"(device={DEVICE})")
    embedder = SentenceTransformer(CONFIG["embedding_model_name"], device=DEVICE)
    print("✅ Embedding model ready")


Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 182 chunks...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

✅ Index saved
Index file: /content/drive/MyDrive/colab_local_llm_kb/index/faiss.index
Metadata file: /content/drive/MyDrive/colab_local_llm_kb/index/metadata.pkl


In [ ]:
# ============================================================
# Cell 10 - Load the local LLM
# Why:
# - llama-cpp-python runs GGUF models locally inside the Colab runtime
# - n_gpu_layers=-1 offloads all layers to GPU when available
# Reference:
# https://github.com/abetlen/llama-cpp-python
# ============================================================

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

use_gpu = TORCH_AVAILABLE and torch.cuda.is_available()
print(f"Loading LLM (GPU offload: {use_gpu})...")

llm = Llama(
    model_path=str(MODEL_PATH),
    n_ctx=CONFIG["n_ctx"],
    n_threads=CONFIG["n_threads"],
    n_gpu_layers=-1 if use_gpu else 0,
    verbose=False,
)

print("✅ Local LLM loaded")
print("Model path:", MODEL_PATH)


✅ Local LLM loaded
Model path: /content/drive/MyDrive/colab_local_llm_kb/models/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf


In [ ]:
# ============================================================
# Cell 11 - Retrieval and prompt assembly helpers
# ============================================================

if "embedder" not in globals() or "index" not in globals():
    raise RuntimeError("Run Cell 9 (build/reload index) before this cell.")

def retrieve_context(query: str, top_k: int = None):
    top_k = top_k or CONFIG["top_k"]
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        item = documents[idx]
        results.append({
            "score": float(score),
            "source": item["source"],
            "chunk_id": item["chunk_id"],
            "text": item["text"],
        })
    return results

def build_prompt(query: str, contexts):
    context_text = "\n\n".join(
        f"[Source: {c['source']} | Chunk: {c['chunk_id']}]\n{c['text']}" for c in contexts
    )
    prompt = f"""You are a helpful assistant answering questions only from the provided knowledge base context.

Rules:
- Use the provided context.
- If the answer is not in the context, say you could not find it in the knowledge base.
- Cite the source filenames in your answer when possible.

Context:
{context_text}

Question:
{query}

Answer:"""
    return prompt

def ask_kb(query: str, top_k: int = None, max_tokens: int = None, temperature: float = None):
    top_k = top_k or CONFIG["top_k"]
    max_tokens = max_tokens or CONFIG["max_tokens"]
    temperature = CONFIG["temperature"] if temperature is None else temperature

    contexts = retrieve_context(query, top_k=top_k)
    prompt = build_prompt(query, contexts)

    response = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        stop=["<|eot_id|>", "\n\nQuestion:"],
    )

    answer = response["choices"][0]["text"].strip()
    return {
        "query": query,
        "answer": answer,
        "contexts": contexts,
        "prompt": prompt,
    }

print("✅ Retrieval helpers ready")


✅ Retrieval helpers ready


In [ ]:
# ============================================================
# Cell 12 - Ask a question
# Single-click usage:
# 1) Edit the query below
# 2) Run this cell
# ============================================================

query = "What topics are covered in my documents?"

result = ask_kb(query)

print("QUESTION:\n", result["query"])
print("\nANSWER:\n", result["answer"])

print("\nRETRIEVED SOURCES:")
for i, ctx in enumerate(result["contexts"], 1):
    print(f"\n{i}. source={ctx['source']} chunk={ctx['chunk_id']} score={ctx['score']:.4f}")
    print(ctx["text"][:400], "...")


QUESTION:
 What topics are covered in my documents?

ANSWER:
 The documents cover topics such as:
- AI Readiness Records
- Knowledge Base
- Requirements
- Metrics
- Granular Priorities
- Contextualization and Regional Impact
- Granular Priorities and Contextualization and Regional Impact
- Granular Priorities and Contextualization and Regional Impact
- Granular Priorities and Contextualization and Regional Impact
- Granular Priorities and Contextualization and Regional Impact
- Granular Priorities and Contextualization and Regional Impact
- Granular Priorities and Contextualization and Regional Impact
- Granular Priorities and Contextualization and Regional Impact
- Granular Priorities and Contextualization and Regional Impact
- Granular Priorities and Contextualization and Regional Impact
- Granular Priorities and Contextualization and Regional Impact
- Granular Priorities and Contextualization and Regional Impact
- Granular Priorities and Contextualization and Regional Impact
- Granu

## Recommended run order

> **Reminder:** these are *logical step numbers* (matching the `# Cell N` comments), not Jupyter cell indices.

For a **fresh Colab session** (`rebuild_index = True`):

1. Run **Cell 2** — install dependencies (triggers auto-restart; expected)
2. Run **Cell 3** — verify imports
3. Run **Cell 4** — review/edit config
4. Run **Cell 5** — mount Google Drive
5. Run **Cell 6** — download the GGUF model
6. Run **Cell 7** — upload docs (or copy files into Drive manually)
7. Run **Cell 8** — parse and chunk documents
8. Run **Cell 9** — build FAISS index (`rebuild_index=True`)
9. Run **Cell 10** — load the local LLM
10. Run **Cell 11** — prepare retrieval helpers
11. Run **Cell 12** — ask questions

For **returning sessions** (index already saved on Drive):

- Set `rebuild_index = False` in Cell 4
- Skip Cell 7 and Cell 8 (no need to re-parse docs)
- Cell 9 will reload the saved index automatically
- Continue from Cell 10 onward
